# CRAFT Phase 2: Curriculum-guided Reinforced Adaptive Fine-Tuning (RL Loop)
This notebook trains the model using GRPO reinforcement learning incorporating:
- Component A: Symbolic Math & NLI Logical Step Verifiers
- Component B: Contrastive Step-Level Preference DPO (starts at step 100)
- Component C: Adaptive Curriculum selecting from the 40-70% difficulty zone

In [ ]:
# 1. Install dependencies
!pip install -q transformers bitsandbytes accelerate datasets peft trl loguru pyyaml scipy numpy sentence-transformers

In [ ]:
# 2. Check SFT Warmup checkpoints and difficulty map exist
import os
assert os.path.exists("checkpoints/sft/final"), "Error: checkpoints/sft/final not found. Run Phase 1 first!"
assert os.path.exists("difficulty_map.json"), "Error: difficulty_map.json not found. Run Phase 0 first!"

In [ ]:
# 3. Diagnostic tests to verify reward scorer and answer extraction
from src.phase2_rl.component_a.reward_scorer import RewardScorer
scorer = RewardScorer()

# Test 1: Check answer extraction
test_response_correct = """
<thought>
Step 1: John has 3 apples.
Step 2: After buying 4 more: 3 + 4 = 7 apples.
Step 3: After giving 2 to Mary: 7 - 2 = 5 apples.
</thought>
<answer>5</answer>
"""
test_question = {"answer": "5"}
score1, success1 = scorer.score_with_success(test_question, test_response_correct)
print(f"Test 1 (correct answer): score={score1:.3f}, success={success1}")
assert success1 == True, "FAIL: answer extraction broken for correct response"
assert score1 > 0.7, f"FAIL: score too low for correct response: {score1}"

# Test 2: Check incorrect answer scores lower
test_response_wrong = "<thought>Step 1: 3 + 4 = 7</thought><answer>3</answer>"
score2, success2 = scorer.score_with_success(test_question, test_response_wrong)
print(f"Test 2 (wrong answer): score={score2:.3f}, success={success2}")
assert success2 == False, "FAIL: wrong answer marked as correct"

# Test 3: Check GSM8K format ground truth parsing
test_question_gsm = {"answer": "Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 eggs per day.\n#### 9"}
score3, success3 = scorer.score_with_success(test_question_gsm, test_response_correct)
print(f"Test 3 (GSM8K format): score={score3:.3f}, success={success3}")
assert success3 == False, "FAIL: 5 != 9 but marked correct"

# Test 4: Verify rewards vary (not all identical)
scores = [score1, score2]
assert scores[0] != scores[1], "FAIL: rewards not varying (will cause zero GRPO loss)"

print("\u2705 ALL DIAGNOSTIC TESTS PASSED. Safe to start training.")
print(f"Score range: {min(scores):.3f} to {max(scores):.3f} (should NOT all be 0.2000)")

In [ ]:
# 4. Run CRAFT RL Loop training
# Checkpoint manager handles saves and resumes if session dies
!python -m src.phase2_rl.craft_rl_loop --config phi3_mini --hardware kaggle --output checkpoints/rl --resume

In [ ]:
# 5. Verify RL checkpoints
final_path = "checkpoints/rl/final"
if os.path.exists(final_path):
    print("CRAFT RL Loop training successfully completed. Weights saved at:", final_path)
    print(os.listdir(final_path))
else:
    print("Error: checkpoints/rl/final not found!")